In [1]:
import os
import sys
import json
import yaml
import torch
import numpy as np
from pathlib import Path
from datetime import datetime
from itertools import combinations
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm 

# --- ROBUST PROJECT ROOT FINDER ---
def find_project_root(current_path, target_folder="src"):
    current_path = Path(current_path).resolve()
    for parent in [current_path] + list(current_path.parents):
        if (parent / target_folder).exists():
            return parent
    return None

# Find the root starting from the current directory
PROJECT_ROOT = find_project_root(Path.cwd())

if PROJECT_ROOT:
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT)) # Insert at 0 to prioritize your src
    print(f"✅ Project Root found at: {PROJECT_ROOT}")
else:
    # Manual hard-coded fallback for your specific HPC setup
    PROJECT_ROOT = Path("/scratch/sp7007/MoT-DAQCNN")
    sys.path.insert(0, str(PROJECT_ROOT))
    print(f"⚠️ Manual Path fallback: {PROJECT_ROOT}")

# Now these imports will work
from src.layers.quantum_convolution import QuantumConv2d
from src.utils.color_conversion import rgb_to_grayscale_tensor
from src.utils.data import load_medmnist_dataset

print("🚀 Libraries and Paths loaded.")

✅ Project Root found at: /scratch/sp7007/MoT-DAQCNN


/scratch/sp7007/nyuenv/lib/python3.12/site-packages/pennylane/__init__.py:212: PennyLaneDeprecationWarning: PennyLane v0.44 has dropped maintainence support for NumPy < 2.0.0. You have version 1.26.4 installed. Future versions of PennyLane will not work with NumPy<2.0. Please consider upgrading NumPy using `python -m pip install numpy --upgrade`. 
  warnings.warn(


🚀 Libraries and Paths loaded.


In [2]:
# Load the specific config you requested
CONFIG_PATH = PROJECT_ROOT / "configs" / "breast_mnist" / "cache_generation" / "digital_zz.yml"

with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

# Detection
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    # lightning.gpu is the high-performance CUDA backend for PennyLane
    QUANTUM_DEVICE = "lightning.gpu"
    INTERFACE = "autograd"
    print(f"🔥 GPU ACTIVE: {torch.cuda.get_device_name(0)}")
else:
    QUANTUM_DEVICE = "lightning.qubit"
    INTERFACE = "torch"
    print("🧊 NO GPU: Using CPU (This will take a long time)")

# Extract settings from YAML
dataset_cfg = config.get("dataset", {})
model_cfg = config.get("model", {})

# Override the quantum device from config with our detected high-speed one
model_cfg["quantum_device"] = QUANTUM_DEVICE

🔥 GPU ACTIVE: NVIDIA A100-PCIE-40GB


In [3]:
print(f"🏗️ Building Quantum Layer with {len(model_cfg['kernel_topology_names'])} topologies...")

q_conv = QuantumConv2d(
    in_channels=1 if dataset_cfg['color_space'] == "GRAYSCALE" else 3,
    kernel_size=model_cfg['kernel_size'],
    stride=model_cfg['stride'],
    kernel_topology_names=model_cfg['kernel_topology_names'],
    scaling_factor=model_cfg['scaling_factor'],
    evolution_time=model_cfg['evolution_time'],
    mode=model_cfg.get("mode", "trotter"),
    quantum_device=QUANTUM_DEVICE,
    interface=INTERFACE,
    include_correlators=model_cfg.get("include_correlators", True),
    encoding_mode=model_cfg.get("encoding_mode", "digital"),
)

q_conv.to(device)
q_conv.eval()

print(f"✅ Quantum layer initialized. Output channels: {q_conv.out_channels}")

🏗️ Building Quantum Layer with 4 topologies...
✅ Quantum layer initialized. Output channels: 180


In [ ]:
data_root = PROJECT_ROOT / "data"
results = {}

for split in ["train", "val", "test"]:
    ds = load_medmnist_dataset(dataset_cfg['name'], split, data_root)
    # num_workers must be 0 for GPU quantum simulations to avoid context errors
    loader = DataLoader(ds, batch_size=dataset_cfg['batch_size'], shuffle=False, num_workers=0)
    
    all_features = []
    all_labels = []

    for images, labels in tqdm(loader, desc=f"Processing {split}"):
        images = images.to(device)
        
        with torch.no_grad():
            if dataset_cfg['color_space'] == "GRAYSCALE" and images.shape[1] == 3:
                images = rgb_to_grayscale_tensor(images)
            
            # The actual quantum math happens here
            q_out = q_conv(images)
            
        all_features.append(q_out.cpu().numpy())
        
        lbl = labels.numpy()
        if lbl.ndim > 1: lbl = lbl.squeeze(-1)
        all_labels.append(lbl)

    results[f"{split}_features"] = np.concatenate(all_features, axis=0)
    results[f"{split}_labels"] = np.concatenate(all_labels, axis=0)
    print(f"✓ {split} complete: {results[f'{split}_features'].shape}")

Using downloaded and verified file: /scratch/sp7007/MoT-DAQCNN/data/breastmnist.npz


Processing train:   0%|          | 0/69 [00:00<?, ?it/s]

In [ ]:
# Generate descriptive filename
topo_short = "-".join([t[:3] for t in model_cfg['kernel_topology_names']])
filename = f"{dataset_cfg['name']}__k{model_cfg['kernel_size']}_s{model_cfg['stride']}_t{topo_short}_zz.npz"

output_dir = data_root / "quantum_datasets"
output_dir.mkdir(parents=True, exist_ok=True)
save_path = output_dir / filename

metadata = {
    "config": config,
    "total_channels": q_conv.out_channels,
    "created_at": datetime.now().isoformat(),
    "device_used": str(device)
}

np.savez_compressed(
    save_path,
    train_features=results["train_features"],
    train_labels=results["train_labels"],
    val_features=results["val_features"],
    val_labels=results["val_labels"],
    test_features=results["test_features"],
    test_labels=results["test_labels"],
    metadata=json.dumps(metadata)
)

print(f"✨ SUCCESS! Dataset saved to: {save_path}")